# This is a PCA_MLP evaluation script 
Features Extracted using PCA and classified using MLP

## Prepare and load data

In [37]:
import pandas as pd
import numpy as np
import scipy.io as sio
import torch
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import PCA_features


In [38]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

display(df)

,0,1,2,3,4,5,6,7,8,9,...,35991,35992,35993,35994,35995,35996,35997,35998,35999,y
0,6.420612e-04,0.000633,0.000670,0.000698,0.000701,0.000600,0.000681,0.000659,0.000620,0.000591,...,0.002003,0.001969,0.001982,0.002026,0.002025,0.002084,0.002075,0.002127,0.002111,0
1,4.241467e-04,0.000389,0.000357,0.000275,0.000292,0.000398,0.000435,0.000499,0.000531,0.000534,...,0.001543,0.001660,0.001608,0.001603,0.001649,0.001621,0.001581,0.001594,0.001680,0
2,6.834269e-04,0.000704,0.000699,0.000687,0.000640,0.000681,0.000697,0.000672,0.000621,0.000615,...,0.000679,0.000784,0.000785,0.000791,0.000791,0.000867,0.000952,0.000950,0.000922,0
3,-5.584955e-04,-0.000551,-0.000609,-0.000610,-0.000565,-0.000661,-0.000465,-0.000538,-0.000598,-0.000554,...,-0.000068,-0.000020,-0.000009,-0.000013,0.000010,0.000079,0.000041,0.000134,0.000170,0
4,7.988214e-04,0.000801,0.000856,0.000838,0.000846,0.000783,0.000760,0.000799,0.000845,0.000833,...,0.000793,0.000758,0.000798,0.000798,0.000824,0.000797,0.000851,0.000828,0.000918,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3304,1.330376e-04,0.000131,0.000008,-0.000098,-0.000104,-0.000230,-0.000071,0.000113,0.000143,0.000275,...,0.000149,0.000202,0.000195,0.000149,0.000053,0.000130,0.000085,0.000021,0.000029,0
3305,-2.276897e-05,-0.000009,0.000100,0.000051,0.000111,0.000186,0.000232,0.000241,0.000159,0.000140,...,0.000003,0.000035,0.000009,0.000116,0.000034,0.000073,0.000070,0.000050,0.000146,0
3306,9.083748e-05,-0.000041,-0.000091,-0.000148,-0.000211,-0.000165,-0.000114,-0.000016,0.000076,-0.000026,...,-0.000103,-0.000093,-0.000066,-0.000094,0.000037,0.000025,-0.000050,-0.000195,-0.000179,0
3307,-9.536743e-07,-0.000116,-0.000089,-0.000092,-0.000116,-0.000047,-0.000134,0.000015,-0.000057,-0.000111,...,0.000074,0.000167,0.000184,0.000231,0.000162,0.000164,0.000036,0.000082,0.000246,0


In [88]:
import torch.utils


shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data_P1 = PCA_features(df_X, df_Y, n_components=1000)

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3308))

train_data = torch.utils.data.Subset(data_P1, train_idx)
test_data = torch.utils.data.Subset(data_P1, test_idx)

batch_size = 1

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)


In [89]:
input_dim = data_P1[0][0].shape
print("Input dimension: ", input_dim)

Input dimension:  torch.Size([1000])


In [90]:
def init_train_var(model):
  criterion = torch.nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=0.0001, weight_decay=0.0001)

  return criterion, optimizer

In [91]:
from classification_models import classification

def train(train_loader, batch_size, epochs):
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

  model = classification.MLP(nb_hidden=128, input_dim=input_dim[0], output_dim=2)
  model.to(device)
  
  criterion, optimizer = init_train_var(model=model)
  history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
  train_steps = len(train_loader.dataset) // batch_size
#   val_steps = len(val_loader.dataset) // batch_size

  nb_epochs = epochs
  # run for nb_epochs
  for e in range(nb_epochs):
      # set the model in training mode
      model.train()
      # initialize the total training and validation loss
      epoch_train_loss = 0
    #   epoch_val_loss = 0
      # initialize the number of correct predictions in the training
      # and validation step
      train_correct = 0
    #   val_correct = 0

      for x, y in train_loader:
          
          x, y = x.to(device), y.to(device)
          optimizer.zero_grad()

          pred = model(x)
          loss = criterion(pred, y)
          loss.backward()
          optimizer.step()

          # add the loss to the total training loss so far and
          # calculate the number of correct predictions
          epoch_train_loss += loss
          train_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    #   # switch off autograd for validation
    #   with torch.no_grad():
    #       # set the model in evaluation mode
    #       model.eval()
    #       # loop over the validation set
    #       for (x, y) in val_loader:

    #           x, y = x.to(device), y.to(device)
    #           pred = model(x)
    #           loss = criterion(pred, y)
    #           epoch_val_loss += loss
    #           val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()



      # calculate the average epoch training and validation loss
      mean_train_loss = epoch_train_loss / train_steps
    #   mean_val_loss = epoch_val_loss / val_steps
      # calculate the training and validation accuracy
      train_correct = train_correct / len(train_loader.dataset)
      # val_correct = val_correct / len(val_loader.dataset)
      # update our training history
      history["train_loss"].append(mean_train_loss.cpu().detach().numpy())
      history["train_acc"].append(train_correct)
    #   history["val_loss"].append(mean_val_loss.cpu().detach().numpy())
      # history["val_acc"].append(val_correct)
      # print the model training and validation information
      print("[INFO] EPOCH: {}/{}".format(e + 1, nb_epochs))
      print("Train loss: {:.6f}, Train accuracy: {:.4f}".format(
          mean_train_loss, train_correct))
    #   print("Val loss: {:.6f}, Val accuracy: {:.4f}\n".format(
    #       mean_val_loss, val_correct))
      # save the model if the validation loss is less than the previous
      # if mean_val_loss - prev_mean_val_loss> 0.01:
      #   break
      # else:
      #   prev_mean_val_loss = mean_val_loss

  torch.save(model, "pca_mlp.pth")

In [92]:
def test(model_path, test_loader):
  # test on the test set
  print("[INFO] Testing the model")
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  model = torch.load(model_path,weights_only=False)
  model.to(device)
  test_correct = 0
  with torch.no_grad():
      model.eval()
      for x, y in test_loader:
          x, y = x.to(device), y.to(device)
          pred = model(x)
          test_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_acc = test_correct / len(test_loader.dataset)
  print(f"Test accuracy: {test_acc:.4f}")

In [105]:
train(train_loader, batch_size=1, epochs=20)

[INFO] EPOCH: 1/20
Train loss: 0.579486, Train accuracy: 0.6717
[INFO] EPOCH: 2/20
Train loss: 0.476874, Train accuracy: 0.7530
[INFO] EPOCH: 3/20
Train loss: 0.428981, Train accuracy: 0.7970
[INFO] EPOCH: 4/20
Train loss: 0.389428, Train accuracy: 0.8327
[INFO] EPOCH: 5/20
Train loss: 0.354975, Train accuracy: 0.8567
[INFO] EPOCH: 6/20
Train loss: 0.324225, Train accuracy: 0.8760
[INFO] EPOCH: 7/20
Train loss: 0.295880, Train accuracy: 0.9013
[INFO] EPOCH: 8/20
Train loss: 0.269667, Train accuracy: 0.9160
[INFO] EPOCH: 9/20
Train loss: 0.247291, Train accuracy: 0.9270
[INFO] EPOCH: 10/20
Train loss: 0.225719, Train accuracy: 0.9390
[INFO] EPOCH: 11/20
Train loss: 0.206805, Train accuracy: 0.9460
[INFO] EPOCH: 12/20
Train loss: 0.189432, Train accuracy: 0.9533
[INFO] EPOCH: 13/20
Train loss: 0.173874, Train accuracy: 0.9613
[INFO] EPOCH: 14/20
Train loss: 0.159774, Train accuracy: 0.9667
[INFO] EPOCH: 15/20
Train loss: 0.147164, Train accuracy: 0.9707
[INFO] EPOCH: 16/20
Train loss: 0.

In [106]:
test("pca_mlp.pth", test_loader)

[INFO] Testing the model
Test accuracy: 0.8442
